## Step 1: Heuristic Segmentation

In [ ]:
import json
import re
from typing import List
import pandas as pd


def split_text_into_paragraphs(text: str) -> List[str]:
    # Normalize line breaks
    text = text.replace("\\n", "\n")
    heuristic_patterns = [
        r"\bHmm\b",
        r"\bWait\b",
        r"\bBut\b",
        r"\bSo\b",
        r"\bAlternatively\b",
        r"\bNow\b",
    ]

    # Split text into sentences
    sentences = re.split(r'(?<=[\.\?\!])\s+', text)
    paragraphs = []
    current_paragraph = []

    for i, sentence in enumerate(sentences):
        stripped = sentence.strip()
        is_trigger = any(re.search(p, stripped, re.IGNORECASE) for p in heuristic_patterns)

        # Only split if:
        # - pattern triggers
        # - current paragraph has at least 3 sentences
        if is_trigger and len(current_paragraph) >= 4:
            paragraphs.append(" ".join(current_paragraph).strip())
            current_paragraph = []

        current_paragraph.append(stripped)

    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph).strip())

    numbered_paragraphs = [f"N{i + 1}: {para}" for i, para in enumerate(paragraphs)]
    return numbered_paragraphs


def process_df_column_to_jsonl(
    df: pd.DataFrame,
    column: str,
    output_file: str,
    id_col: str | None = None,
):
    """
    Args:
        df: input dataframe
        column: text column to process
        output_file: jsonl output path
        id_col: optional id column name (otherwise use index)
    """

    df = df.copy()

    # 1. deal with NaN
    df[column] = df[column].fillna("")

    # 2. split paragraphs
    df["paragraphs"] = df[column].apply(split_text_into_paragraphs)

    # 3. id 
    if id_col is not None:
        df["id"] = df[id_col]
    else:
        df = df.reset_index(drop=True)
        df["id"] = df.index + 1

    # 4. select columns
    out_df = df[["id", "paragraphs",'question','target','no_last_thinking_sentence','last_thinking_sentence']]

    # 5. dump jsonl
    out_df.to_json(
        output_file,
        orient="records",
        lines=True,
        force_ascii=False,
    )

In [ ]:
df_merged_best_seed_filtered = pd.read_json('./data/self_distill_best_of_N_seed.jsonl',lines=True)
df_merged_best_seed_filtered['no_last_thinking_sentence'] = df_merged_best_seed_filtered['last_thinking_sentence'].apply(lambda x: x=='')
process_df_column_to_jsonl(df_merged_best_seed_filtered,'think','self_distill_best_of_N_seed_think_heuristic_segmentation.jsonl')

## Step2: Taxonomy Generation

In [ ]:
df_self_distill_best_of_N_seed_think_heuristic_segmentation = pd.read_json('self_distill_best_of_N_seed_think_heuristic_segmentation.jsonl',lines=True)

In [ ]:
def build_taxonomy_prompt(row):
    paragraphs = row["paragraphs"]
    prompt = f"""You will be given a list of reasoning segment nodes, each labeled (e.g., N1, N2, …), representing steps in a reasoning trace.

Your task has to strictly follow three steps:
First, Analyze each node: Carefully read all reasoning nodes and provide a brief analysis of each node’s function in the reasoning process.
Second, Assign reasoning strategies: Using the taxonomy provided below, identify the primary reasoning strategy each node represents. Some longer nodes may involve multiple strategies—if so, also indicate a secondary strategy. If no secondary applies, leave it as “None”.
Finally, Strictly Format your output as jsonl as below: 

```jsonl

{{"id": "N1", "taxonomy_primary_type": "verification", "taxonomy_secondary_type": None}}
{{"id": "N2", "taxonomy_primary_type": "verification", "taxonomy_secondary_type": "Exploration"}}
...
```

Reasoning Taxonomy

There are 5 types of reasoning strategies:
Backtracking:
The node has to revisit and modify a previous step or assumption to correct an error, resolve a conflict, or incorporate a new insight that alters the reasoning path. It explicitly retracts or adjusts earlier reasoning to improve accuracy or explore a different approach.

Verification:
The node has to test or confirm the correctness a specific claim, assumption, or result without modifying the reasoning path. 

Exploration:
The node proposes new hypotheses, possibilities, or approaches to the problem in an open-ended manner, without committing to a definitive solution. It reflects divergent thinking aimed at uncovering patterns, options, or insights.

Clarification:
The node rephrases, restates, or defines terms, assumptions, or problem constraints to reduce ambiguity and enhance understanding. It establishes a clear foundation for further reasoning without advancing the solution directly.

Conclusion:
The node synthesizes prior reasoning to assert a final or intermediate solution, judgment, or result. It consolidates insights to resolve a reasoning path or subpath, often with confidence.

Node:{paragraphs}
"""
    return prompt


Preparing data for google Gemini batch API

In [ ]:
import json
def create_batch_requests_file(df, output_filename, prompt_builder=build_taxonomy_prompt):
    """
    Generate a request for each row in the DataFrame and save them as a JSONL file.
    """
    requests = []
    
    for idx, row in df.iterrows():

        prompt_text = prompt_builder(row)

        request_obj = {
            "key": f"request-{idx+1}",
            "request": {
                "contents": [{
                    "role": "user", # It's good practice to explicitly set the role
                    "parts": [{
                        "text": prompt_text
                    }]
                }],
                # --- ADD THE CONFIG HERE ---
                "generationConfig": {
                    "temperature": 0
                }
                # ---------------------------
            }
        }
        requests.append(request_obj)
    
    with open(output_filename, "w", encoding="utf-8") as f:
        for req in requests:
            f.write(json.dumps(req, ensure_ascii=False) + "\n")
    
    print(f"Generate {len(requests)} requests file {output_filename}")
    return requests   

In [ ]:
import pandas as pd
import math
import json
import os
df = pd.read_json('self_distill_best_of_N_seed_think_heuristic_segmentation.jsonl',lines=True)

def split_batches(df, output_dir, prefix="my-batch-request-taxonomy"):
    os.makedirs(output_dir, exist_ok=True)

    total = len(df)
    batch_size = 100
    num_batches = math.ceil(total / batch_size)  # 997 -> 100

    for i in range(num_batches):
        start = i * batch_size
        end = min((i + 1) * batch_size, total)

        batch_df = df.iloc[start:end]

        output_path = os.path.join(
            output_dir,
            f"{prefix}-{i+1}.jsonl"   # e.g., my-batch-requests-1.jsonl
        )

        create_batch_requests_file(batch_df, output_path)
        print(f"Saved batch {i+1}: rows {start}–{end-1} -> {output_path}")
split_batches(df, './data/batch_requests_taxonomy')

**Need to Use google_batch_api.ipynb to get taxnomy results: Results from batch requests**

In [ ]:
import pandas as pd

base = "./taxonomy_batches"
# Results from batch requests
dfs = [
    pd.read_json(f"{base}/my-batch-request-taxonomy-{i}.jsonl_result.jsonl", lines=True)
    for i in range(1, 10)
]

In [ ]:
import re

def extract_jsonl(text):
    # match ```jsonl ... ``` 
    pattern = r"```jsonl\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

In [ ]:

df_merged = pd.concat(dfs, ignore_index=True)
df_merged['id'] = df_merged['key'].apply(lambda x: int(str(x).split('-')[-1]))
df_merged['jsonl'] = df_merged['response'].apply(
    lambda x: extract_jsonl(
        x['candidates'][0]['content']['parts'][0]['text']
    )
)
df_merged.drop('key', axis=1, inplace=True)
df_merged = df_merged.rename(columns={"jsonl": "taxonomy"})
df_merged = df_merged.rename(columns={"response": "taxonomy_response"})

df_merged.to_json('./data/batch-data-all-taxonomy.jsonl', orient='records', lines=True)

In [ ]:
df_result = pd.merge(df_merged, df_self_distill_best_of_N_seed_think_heuristic_segmentation, on='id', how='left')
df_filtered = df_result[
    df_result["taxonomy_response"].notna() &
    df_result["taxonomy"].notna()
]
df_filtered.to_json('self_filter_all_taxonomy.jsonl', orient='records', lines=True)

## Step 3: Find Conclusion Nodes

In [ ]:
all_taxonomy = pd.read_json('./data/self_filter_all_taxonomy.jsonl',lines=True)
df_all_taxonomy =all_taxonomy[['paragraphs','taxonomy','question','target']]
output_dir = "./data/conclusion_batches"
split_batches(df_all_taxonomy, output_dir, prefix="my-batch-request-conclusion")

Preparing Data for google batch api to get Conclusion Results

In [ ]:
import pandas as pd
import ast
import json

import ast

import json
import ast

def parse_taxonomy_str_safe_check(taxonomy_str: str):
    """
    Safely parse a taxonomy string:
    - Supports JSON or Python literals
    - Skips lines that fail to parse
    - Raises an exception if all lines in the taxonomy fail to parse
    """
    result = []

    for line in str(taxonomy_str).strip().splitlines():
        line = line.strip()
        if not line:
            continue

        parsed = None
        # Try JSON parsing
        try:
            parsed = json.loads(line)
        except Exception:
            pass

        # Try Python literal parsing
        if parsed is None:
            try:
                parsed = ast.literal_eval(line)
            except Exception:
                # Skip this line if parsing fails
                continue

        if isinstance(parsed, dict):
            result.append(parsed)

    if not result:
        raise ValueError("❌ taxonomy parse failed: all lines invalid")

    return result


def extract_conclusion_ids(taxonomy_list):

    """
    Extract the IDs (ignoring case) of all conclusions from the taxonomy column (list of dict).
    """
    conclusion_ids = []

    for item in taxonomy_list:
        primary = item.get("taxonomy_primary_type", "")
        if primary.lower() == "conclusion":
            conclusion_ids.append(item.get("id"))

    return conclusion_ids

def build_prompt_find_conclusion(row):
    # extract conclusion id from row
    taxonomy_raw = row["taxonomy"]          #   str
    taxonomy_list = parse_taxonomy_str_safe_check(taxonomy_raw)
    conclusion_ids = extract_conclusion_ids(taxonomy_list)
    
    # Format it as a string, for example: ['N5','N8','N11']
    conclusion_id_str = str(conclusion_ids)
 
    question = row["question"]
    target = row["target"]
    paragraphs = row["paragraphs"]  # reasoning trace
    prompt = f"""
Task:
You are given a reasoning trace divided into nodes (e.g., N1, N2, …). 
Each node represents a step in the reasoning process. 
You are also given a list of conclusion nodes:{conclusion_id_str} identified by their serial numbers.

For each conclusion node, do the following **step by step**, showing your complete reasoning trace:

1. Determine whether the conclusion node matches the Correct Answer.
   - If the conclusion exactly matches the Correct Answer, mark it as 1 (correct).
   - If it does not match, mark it as 0 (incorrect).
   
2. Determine the type of the conclusion node:
   - **Intermediate Conclusion**: a step that supports later reasoning but does not directly answer the question.
   - **Answering Conclusion**: a conclusion that is intended to answer the question, regardless of whether it is fully correct or ultimately the final answer.

3. Include the question and the Correct Answer:
   - Question: {question}  
   - Correct Answer: {target}

4. Export the result in JSONL format in one JSONL file, including the node id, correctness, and type. Example:

```jsonl
{{"conclusion_node": "N5", "is_correct": 0, "type": "Intermediate Conclusion"}}
{{"conclusion_node": "N10", "is_correct": 1, "type": "Answering Conclusion"}}
{{"conclusion_node": "N12", "is_correct": 1, "type": "Answering Conclusion"}}
Complete Nodes:{paragraphs}
""" 
    return prompt

In [ ]:
import os, math
def split_batches(df, output_dir, prefix="my-batch-request-taxonomy"):
    os.makedirs(output_dir, exist_ok=True)

    total = len(df)
    batch_size = 100
    num_batches = math.ceil(total / batch_size)  # 997 -> 10

    for i in range(num_batches):
        start = i * batch_size
        end = min((i + 1) * batch_size, total)

        batch_df = df.iloc[start:end]

        output_path = os.path.join(
            output_dir,
            f"{prefix}-{i+1}.jsonl"   # e.g., my-batch-requests-1.jsonl
        )

        create_batch_requests_file(batch_df, output_path,build_prompt_find_conclusion)
        print(f"Saved batch {i+1}: rows {start}–{end-1} -> {output_path}")

In [ ]:
all_taxonomy = pd.read_json('self_filter_all_taxonomy.jsonl',lines=True)
df_all_taxonomy =all_taxonomy[['paragraphs','taxonomy','question','target']]

output_dir = "./data/conclusion_batches"
split_batches(df_all_taxonomy, output_dir, prefix="my-batch-request-conclusion")
# create_batch_requests_file(all_taxonomy)
# Exampleprompt
print("\nOne prompt example:")
print(build_prompt_find_conclusion(all_taxonomy.iloc[21]))

In [ ]:
create_batch_requests_file(df_all_taxonomy)

## Step3: Use Result Conclusion Nodes to prune

In [ ]:
base = "./data/conclusion_batches"

dfs = [
    pd.read_json(
        f"{base}/my-batch-request-conclusion-{i}.jsonl_result.jsonl",
        lines=True
    )
    for i in range(1, 10)
]

df_all = pd.concat(dfs, ignore_index=True)

In [ ]:
df_merged['id'] = df_merged['key'].apply(lambda x: int(str(x).split('-')[-1]))
df_merged.drop('key', axis=1, inplace=True)

import re
df_merged['jsonl'] = df_merged['response'].apply(
    lambda x: extract_jsonl(
        x['candidates'][0]['content']['parts'][0]['text']
    )
)



In [ ]:
raw1 =  pd.read_json('/Users/dexter/PycharmProjects/pythonProject/CoT/CoT_code/Datapipeline/self_distill/bestofN/self_distill_best_of_N_seed.jsonl',lines=True)
# df =test[['paragraphs','taxonomy']]
# df
raw1['id'] = range(1, len(raw1) + 1)
raw1['no_last_thinking_sentence'] = raw1['last_thinking_sentence'].apply(lambda x: x=='')
raw2 = pd.read_json('self_distill_best_of_N_seed_think_heuristic_segmentation.jsonl',lines=True)
raw2_subset = raw2[['question', 'paragraphs']]
raw = pd.merge(raw1, raw2_subset, on='question', how='left')

df_result = pd.merge(df_merged, raw, on='id', how='left')
df_result['total_nodes'] = df_result['paragraphs'].apply(lambda x: len(x))

Find first correct answering node

In [ ]:
import json

def find_first_correct_answering(jsonl_text):
    """
    Input：jsonl text
    Output：first conclusion_node，if not exist return None
    """
    if not isinstance(jsonl_text, str):
        return None
    
    jsonl_text = jsonl_text.strip()
    if jsonl_text == "":
        return None

    for line in jsonl_text.split("\n"):
        try:
            obj = json.loads(line)
            if obj.get("is_correct") == 1:
                return obj.get("conclusion_node")
        
        except Exception:
            return None  # 非法 json 行直接结束
    
    return None

In [ ]:
df_result['first_correct_nodes'] = df_result['jsonl'].apply(lambda x: find_first_correct_answering(x))
df_result = df_result[df_result['first_correct_nodes'].notna()]
df_result['first_node_index'] = df_result['first_correct_nodes'].apply(lambda x: int(x.replace('N','')))
df_result['need_pruned'] = df_result['total_nodes'] != df_result['first_node_index']

In [ ]:
def process_row(row):
    """
    Generate the final text based on a row from df_result
    """
    # 1️⃣ If pruning is not needed, directly use think + </think> + content
    if not row['need_pruned']:
        return f"{row['think']}</think>{row['content']}"
    
    # 2️⃣ Pruning is required
    # First, retrieve paragraphs
    paragraphs = row.get('paragraphs', [])
    first_n = row.get('first_node_index', 0)
    
    # Keep the first `first_n` elements
    paragraphs_to_use = paragraphs[:first_n]
    
    # Remove markers like N1:, N2:, etc.
    cleaned_paragraphs = []
    for p in paragraphs_to_use:
        # Remove leading Nxx: markers
        cleaned_paragraphs.append(p.split(":", 1)[-1].strip())
    
    # 3️⃣ Decide whether to append last_thinking_sentence based on no_last_thinking_sentence
    no_last = row.get('no_last_thinking_sentence', False)
    last_sentence = ""
    if not no_last:
        last_sentence = row.get('last_thinking_sentence', "").strip()
    
    # 4️⃣ Assemble the final text
    final_text = "\n".join(cleaned_paragraphs)
    if last_sentence:
        final_text += "\n" + last_sentence
    
    final_text += "</think>" + row.get('content', "")
    
    return final_text


# Apply to the entire DataFrame
df_result['processed_text'] = df_result.apply(process_row, axis=1)

In [ ]:
import json

output_file = "./data/self_distill_training_data_pruned.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df_result.iterrows():
        obj = {
            "messages": [
                {"role": "user", "content": row.get("question", "")},
                {"role": "assistant", "content": row.get("processed_text", "")}
            ]
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"File path：{output_file}")